[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/09_causal_attention.ipynb)

# 🔴 Hard: Causal Self-Attention

Implement **causal (masked) self-attention** — the attention used in GPT-style decoders.

Same as softmax attention, but each position can **only attend to itself and earlier positions** (no peeking at future tokens).

$$\text{scores}_{ij} = \begin{cases} \frac{Q_i \cdot K_j}{\sqrt{d_k}} & \text{if } j \le i \\ -\infty & \text{if } j > i \end{cases}$$

### Signature
```python
def causal_attention(Q, K, V):
    # Q, K, V: (batch, seq, d) → output: (batch, seq, d_v)
```

### Rules
- Do **NOT** use `F.scaled_dot_product_attention`
- Position $i$ can only attend to positions $\le i$
- You **may** use `torch.softmax`, `torch.bmm`, `torch.triu`

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 1.3 MB/s eta 0:00:00


In [2]:
import torch
import math

In [40]:
# ✏️ YOUR IMPLEMENTATION HERE

def causal_attention(Q, K, V):

    batch, seq, d = Q.shape[0], Q.shape[1], Q.shape[2]

    mask = torch.triu(
      torch.ones([seq, seq], dtype=torch.bool),
      diagonal=1
    )

    scores = torch.einsum('bpd,bqd->bpq', Q, K) / math.sqrt(d)
    print(Q.shape, V.shape, scores.shape, mask.shape)
    scores = scores.masked_fill(mask, -float('inf'))

    scores = torch.softmax(scores, dim=-1)
    ret = torch.einsum('bpq,bqd->bpd', scores, V)

    return ret

In [41]:
# 🧪 Debug
torch.manual_seed(0)
Q = torch.randn(1, 4, 8)
K = torch.randn(1, 4, 8)
V = torch.randn(1, 4, 8)
out = causal_attention(Q, K, V)
print("Output shape:", out.shape)          # (1, 4, 8)
print("Pos 0 == V[0]?", torch.allclose(out[:, 0], V[:, 0], atol=1e-5))  # should be True

torch.Size([1, 4, 8]) torch.Size([1, 4, 8]) torch.Size([1, 4, 4]) torch.Size([4, 4])
Output shape: torch.Size([1, 4, 8])
Pos 0 == V[0]? True


In [42]:
from torch_judge import check
check('causal_attention')


🧪 Testing: Causal Self-Attention (Hard)
──────────────────────────────────────────────────
torch.Size([2, 6, 16]) torch.Size([2, 6, 16]) torch.Size([2, 6, 6]) torch.Size([6, 6])
  ✅ [1/4] Output shape (2.8ms)
torch.Size([1, 8, 16]) torch.Size([1, 8, 16]) torch.Size([1, 8, 8]) torch.Size([8, 8])
torch.Size([1, 8, 16]) torch.Size([1, 8, 16]) torch.Size([1, 8, 8]) torch.Size([8, 8])
  ✅ [2/4] Future tokens don't affect past (5.4ms)
torch.Size([1, 4, 8]) torch.Size([1, 4, 8]) torch.Size([1, 4, 4]) torch.Size([4, 4])
  ✅ [3/4] First position only sees itself (2.8ms)
torch.Size([2, 4, 8]) torch.Size([2, 4, 8]) torch.Size([2, 4, 4]) torch.Size([4, 4])
  ✅ [4/4] Gradient flow (2.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (13.6ms total)
  Progress saved. Run status() to see your dashboard.

